# Tournament

File: tournament.py

In [ ]:
import sys
import os


In [ ]:
import json
import time
import math
import random
import numpy as np
from typing import Callable, Dict, List, Tuple
from functools import partial

from game.game_utils import one_conn_game


In [ ]:
"""
Round-Robin Tournament Evaluation
===================================
Theo đề tài: round-robin giữa các tác tử, ≥100 ván/cặp,
chia đều quyền đi trước/đi sau, khoảng tin cậy Wilson.

Chạy: python eval/tournament.py
"""

import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))


In [ ]:
def wilson_ci(wins: int, n: int, z: float = 1.96) -> Tuple[float, float]:
    """
    Wilson score confidence interval cho tỷ lệ thắng.
    
    Parameters
    ----------
    wins : int
    n : int  (tổng số ván)
    z : float  (1.96 = 95% CI)
    
    Returns
    -------
    (lower, upper) : Tuple[float, float]
    """
    if n == 0:
        return (0.0, 1.0)
    p = wins / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    margin = z * math.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return (max(0.0, center - margin), min(1.0, center + margin))


In [ ]:
def play_match(agent1_fn: Callable, agent2_fn: Callable,
               n_games: int = 200,
               seed: int = 42,
               n_random_openers: int = 0,
               agent1_kwargs: dict = None,
               agent2_kwargs: dict = None,
               verbose: bool = False) -> Dict:
    """
    Chạy n_games ván giữa hai agent, chia đều đi trước/đi sau.
    
    Parameters
    ----------
    agent1_fn : callable
    agent2_fn : callable
    n_games : int  (nên là số chẵn)
    seed : int
    n_random_openers : int  (0-4 nước đầu ngẫu nhiên theo seed để tránh ván lặp)
    agent1_kwargs, agent2_kwargs : dict
    verbose : bool
    
    Returns
    -------
    dict với: wins, losses, draws, score, wilson_ci, avg_time_ms, p95_time_ms
    """
    if agent1_kwargs is None:
        agent1_kwargs = {}
    if agent2_kwargs is None:
        agent2_kwargs = {}

    random.seed(seed)
    np.random.seed(seed)

    wins, losses, draws = 0, 0, 0
    times_ms = []

    for i in range(n_games):
        t0 = time.perf_counter()

        if i % 2 == 0:
            # agent1 đỏ (đi trước)
            reward, _ = one_conn_game(agent1_fn, agent2_fn,
                                       player1_kwargs=agent1_kwargs,
                                       player2_kwargs=agent2_kwargs)
            if reward == 1:
                wins += 1
            elif reward == -1:
                losses += 1
            else:
                draws += 1
        else:
            # agent2 đỏ (đi trước), agent1 vàng
            reward, _ = one_conn_game(agent2_fn, agent1_fn,
                                       player1_kwargs=agent2_kwargs,
                                       player2_kwargs=agent1_kwargs)
            if reward == 1:
                losses += 1
            elif reward == -1:
                wins += 1
            else:
                draws += 1

        elapsed_ms = (time.perf_counter() - t0) * 1000
        times_ms.append(elapsed_ms)

        if verbose and (i + 1) % 20 == 0:
            print(f"    [{i+1}/{n_games}] W={wins} L={losses} D={draws}")

    score = (wins + 0.5 * draws) / n_games
    ci_lower, ci_upper = wilson_ci(wins, n_games)
    avg_t = float(np.mean(times_ms))
    p95_t = float(np.percentile(times_ms, 95))

    return {
        'wins': wins,
        'losses': losses,
        'draws': draws,
        'n_games': n_games,
        'score': round(score, 4),
        'wilson_ci': (round(ci_lower, 4), round(ci_upper, 4)),
        'avg_time_ms': round(avg_t, 2),
        'p95_time_ms': round(p95_t, 2),
    }


In [ ]:
def round_robin(agents: Dict[str, Callable],
                n_games_per_pair: int = 200,
                seed: int = 42,
                agent_kwargs: Dict[str, dict] = None,
                log_file: str = None,
                verbose: bool = True) -> Dict:
    """
    Tổ chức round-robin giữa tất cả các agent.
    
    Parameters
    ----------
    agents : dict  {name: agent_fn}
    n_games_per_pair : int
    seed : int
    agent_kwargs : dict  {name: kwargs_dict}
    log_file : str  (path để lưu JSON log)
    verbose : bool
    
    Returns
    -------
    dict kết quả: matrix và ranking
    """
    if agent_kwargs is None:
        agent_kwargs = {name: {} for name in agents}

    names = list(agents.keys())
    n = len(names)
    results = {}

    print(f"\n{'='*60}")
    print(f"ROUND-ROBIN TOURNAMENT")
    print(f"Agents: {names}")
    print(f"Games per pair: {n_games_per_pair}")
    print(f"{'='*60}\n")

    pairs = [(i, j) for i in range(n) for j in range(i+1, n)]
    total_pairs = len(pairs)

    for idx, (i, j) in enumerate(pairs):
        name1, name2 = names[i], names[j]
        fn1, fn2 = agents[name1], agents[name2]
        kw1 = agent_kwargs.get(name1, {})
        kw2 = agent_kwargs.get(name2, {})

        print(f"[{idx+1}/{total_pairs}] {name1} vs {name2} ...")
        t0 = time.perf_counter()
        match_result = play_match(fn1, fn2, n_games=n_games_per_pair,
                                   seed=seed,
                                   agent1_kwargs=kw1,
                                   agent2_kwargs=kw2,
                                   verbose=verbose)
        elapsed = time.perf_counter() - t0
        print(f"  → score={match_result['score']:.3f}, "
              f"W={match_result['wins']} L={match_result['losses']} D={match_result['draws']}, "
              f"CI={match_result['wilson_ci']}, "
              f"time={elapsed:.1f}s\n")

        results[(name1, name2)] = match_result

    # Tính tổng điểm
    scores = {name: 0.0 for name in names}
    for (n1, n2), res in results.items():
        scores[n1] += res['score']
        scores[n2] += 1.0 - res['score']  # điểm của n2 là 1 - score của n1

    ranking = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    print("\n" + "="*60)
    print("FINAL RANKING")
    print("="*60)
    for rank, (name, score) in enumerate(ranking, 1):
        print(f"  {rank}. {name}: {score:.3f}")

    output = {
        'agents': names,
        'n_games_per_pair': n_games_per_pair,
        'seed': seed,
        'results': {f"{k[0]} vs {k[1]}": v for k, v in results.items()},
        'ranking': ranking,
        'scores': scores,
    }

    if log_file:
        os.makedirs(os.path.dirname(log_file), exist_ok=True)
        with open(log_file, 'w', encoding='utf-8') as f:
            json.dump(output, f, indent=2, ensure_ascii=False, default=str)
        print(f"\nLog đã lưu tại: {log_file}")

    return output


In [ ]:
if __name__ == '__main__':
    from agents.random_agent import conn_random
    from agents.heuristic_agent import conn_think1, conn_think2, conn_think3
    from agents.alphabeta_agent import alphabeta_agent, minimax_eval_agent
    from agents.mcts_agent import mcts_agent
    from functools import partial

    agents = {
        'Random':    conn_random,
        'Think1':    conn_think1,
        'Think2':    conn_think2,
        'Think3':    conn_think3,
        'AlphaBeta': partial(alphabeta_agent, depth=5),
        'MCTS-200':  partial(mcts_agent, n_simulations=200),
    }

    log_path = os.path.join(os.path.dirname(os.path.dirname(__file__)),
                             'logs', 'tournament_results.json')

    results = round_robin(
        agents=agents,
        n_games_per_pair=100,
        seed=42,
        log_file=log_path,
        verbose=False,
    )
